# Day 1: Solutions

Reference solutions, with notes on *why* each one is written the Pythonic way. Run all cells: every test should pass.

---
## Q1. Truthiness check  ·  _Warm-up_

In JS, `[]` and `{}` are truthy. In Python they're not.

**Predict first:** which of these are truthy? Write your guesses in the markdown cell, then run the cell to check.

**Then write** `count_truthy(values)`, which returns how many items in `values` are truthy.

In [ ]:
values = [0, "", [], {}, None, "0", " ", [0], 0.0, -1, "False", set()]
for v in values:
    print(repr(v), "->", bool(v))

In [ ]:
def count_truthy(values):
    return sum(1 for v in values if v)


# Also fine: True counts as 1 when you add booleans
# return sum(map(bool, values))

In [ ]:
assert count_truthy([0, "", [], {}, None, "0", " ", [0], 0.0, -1, "False", set()]) == 5
assert count_truthy([]) == 0
assert count_truthy([[], [[]]]) == 1
print("✅ Q1 passed")

**Why this way**

- `if v` uses truthiness directly. Never write `if v == True` or `if len(v) > 0`.
- The truthy ones are `"0"`, `" "`, `[0]`, `-1` and `"False"`. Non-empty strings are truthy whatever they say, and a list holding a falsy item is still non-empty.
- `sum(1 for v in values if v)` is a *generator expression* (Day 25). A plain loop with a counter is fine too.

---
## Q2. `is` vs `==`  ·  _Warm-up_

**Predict first:** what does each line print?

**Then write** `describe(x)`, which returns:
- `"missing"` if `x` is `None`
- `"empty"` if `x` is falsy but not `None` (`0`, `""`, `[]`, `False`, ...)
- `"value"` otherwise

In [ ]:
a = [1, 2]
b = [1, 2]
c = a
print(a == b)
print(a is b)
print(c is a)
c.append(3)
print(a)
print(1 == "1")
print(1 == 1.0)

In [ ]:
def describe(x):
    if x is None:
        return "missing"
    if not x:
        return "empty"
    return "value"

In [ ]:
assert describe(None) == "missing"
assert describe(0) == "empty"
assert describe("") == "empty"
assert describe([]) == "empty"
assert describe(False) == "empty"
assert describe("hi") == "value"
assert describe([0]) == "value"
print("✅ Q2 passed")

**Why this way**

- `a == b` is `True` (same contents), but `a is b` is `False` (two different list objects). `c = a` doesn't copy, so `c is a` and appending through `c` changes `a`. Same as JS references.
- `1 == "1"` is `False`: Python's `==` never coerces types, so it already behaves like `===`. `1 == 1.0` is `True` because both are numbers with the same value.
- Use `is None`, not `== None`. `None` is a single object, and a class can override `==`.
- Check `None` first, because `None` is falsy too. Order matters.

---
## Q3. Translate JS to Python  ·  _Warm-up_

Translate this JS into Python. `user` is a dict like `{"name": "Asha", "is_admin": True, "messages": ["hi"]}`; `name` and `is_admin` may be missing.

```js
function greet(user) {
  const name = user.name || "stranger";
  const title = user.isAdmin ? "Admin" : "User";
  const count = user.messages.length;
  return `Hello ${name} (${title}), you have ${count} new message${count === 1 ? "" : "s"}`;
}
```

In [ ]:
def greet(user):
    name = user.get("name") or "stranger"
    title = "Admin" if user.get("is_admin") else "User"
    count = len(user["messages"])
    plural = "" if count == 1 else "s"
    return f"Hello {name} ({title}), you have {count} new message{plural}"

In [ ]:
assert greet({"name": "Asha", "is_admin": True, "messages": ["hi"]}) == "Hello Asha (Admin), you have 1 new message"
assert greet({"name": "", "messages": []}) == "Hello stranger (User), you have 0 new messages"
assert greet({"messages": ["a", "b"]}) == "Hello stranger (User), you have 2 new messages"
print("✅ Q3 passed")

**Why this way**

- `user["name"]` raises `KeyError` if the key is missing; `user.get("name")` returns `None`, like JS giving `undefined`.
- `x or default` works exactly like JS `x || default`, including treating `""` as missing.
- The ternary reads in English order: `value_if_true if condition else value_if_false`.
- Pulling `plural` out keeps the f-string readable. You *can* nest a conditional inside `{}`, but it gets hard to read.

---
## Q4. Receipt with f-string formatting  ·  _Core_

Write `format_receipt(items)` where `items` is a list of `(name, unit_price, qty)` tuples. Return one string (lines joined with `"\n"`) laid out like this:

```
Item          Qty      Amount
Coffee          2        7.00
Laptop          1    1,299.99
TOTAL                1,306.99
```

Column rules:
- name: left-aligned, 12 characters wide
- qty: right-aligned, 5 characters wide (blank on the TOTAL row)
- amount (`unit_price * qty`): right-aligned, 12 characters wide, comma thousands separator, 2 decimals

Tip: you can loop over tuples with unpacking: `for name, price, qty in items:`.

In [ ]:
def format_receipt(items):
    lines = [f"{'Item':<12}{'Qty':>5}{'Amount':>12}"]
    total = 0
    for name, price, qty in items:
        amount = price * qty
        total += amount
        lines.append(f"{name:<12}{qty:>5}{amount:>12,.2f}")
    lines.append(f"{'TOTAL':<12}{'':>5}{total:>12,.2f}")
    return "\n".join(lines)

In [ ]:
receipt = format_receipt([("Coffee", 3.5, 2), ("Laptop", 1299.99, 1)])
lines = receipt.split("\n")
assert lines[0] == "Item          Qty      Amount", repr(lines[0])
assert lines[1] == "Coffee          2        7.00", repr(lines[1])
assert lines[2] == "Laptop          1    1,299.99", repr(lines[2])
assert lines[3] == "TOTAL                1,306.99", repr(lines[3])
assert format_receipt([]).split("\n")[-1] == "TOTAL                    0.00"
print(receipt)
print("✅ Q4 passed")

**Why this way**

- The format spec after `:` reads as **[align][width][,][.precision][type]**: `>12,.2f` means right-align, width 12, thousands commas, 2 decimals, fixed-point.
- `<` is left, `>` is right, `^` is centre. Strings default to left and numbers to right, but being explicit is clearer.
- Build a list of lines and `"\n".join(...)` at the end. That's the usual Python way to build multi-line strings, rather than `+=` on a string.
- Tuple unpacking in the `for` header avoids `item[0]`, `item[1]`, `item[2]`.

---
## Q5. Numbers: division and splitting a bill  ·  _Core_

**Predict first:** what does each line print? (One of them raises an error.)

**Then write** `split_bill(total_cents, people)`. It returns a list of `people` integer shares that **add up exactly** to `total_cents`. When it doesn't divide evenly, the first people each pay 1 extra cent. Raise `ValueError` if `people` is less than 1.

Example: `split_bill(1000, 3)` → `[334, 333, 333]`

In [ ]:
print(7 / 2)
print(7 // 2)
print(-7 // 2)
print(-7 % 3)
print(2 ** 100)
print(0.1 + 0.2 == 0.3)
print(int("42") + 1)
print(divmod(17, 5))
try:
    print("3" + 4)
except TypeError as e:
    print("TypeError:", e)

In [ ]:
def split_bill(total_cents, people):
    if people < 1:
        raise ValueError("people must be at least 1")
    base, extra = divmod(total_cents, people)
    return [base + 1] * extra + [base] * (people - extra)

In [ ]:
assert split_bill(1000, 3) == [334, 333, 333]
assert split_bill(900, 3) == [300, 300, 300]
assert split_bill(2, 5) == [1, 1, 0, 0, 0]
assert sum(split_bill(12345, 7)) == 12345
try:
    split_bill(100, 0)
    assert False, "should have raised ValueError"
except ValueError:
    pass
print("✅ Q5 passed")

**Why this way**

- `divmod` gives quotient and remainder in one call, and tuple unpacking names them.
- `[x] * n` repeats a list, and `+` joins lists. That gives "extra people pay base+1, the rest pay base" with no loop.
- Money in **integer cents** avoids float errors (`0.1 + 0.2 != 0.3`, the same as in JS). For decimal money, Python has `decimal.Decimal`.
- `-7 // 2` is `-4` because `//` rounds toward negative infinity, not toward zero like `Math.trunc`. `-7 % 3` is `2` so that `(a // b) * b + a % b == a` always holds.
- `"3" + 4` raises `TypeError`. Python never converts types silently.

---
## Q6. Everything is an object  ·  _Core_

**Predict first:** what does each line print?

**Then write** `apply_all(funcs, value)`, which calls every function in `funcs` with `value` and returns a dict mapping each function's name to its result.

Hint: every function has a `__name__` attribute.

Example: `apply_all([len, str.upper], "hi")` → `{"len": 2, "upper": "HI"}`

In [ ]:
def shout(s):
    return s.upper() + "!"

print(type(42), type("hi"), type(None), type([]))
print(type(shout))
print(type(len))
print(shout.__name__)
yell = shout          # no call: just another name for the same function
print(yell("hey"))
print(yell is shout)
print(type(type))

In [ ]:
def apply_all(funcs, value):
    results = {}
    for f in funcs:
        results[f.__name__] = f(value)
    return results


# One-liner with a dict comprehension (Day 6):
# return {f.__name__: f(value) for f in funcs}

In [ ]:
assert apply_all([len, str.upper, str.title], "hello world") == {
    "len": 11, "upper": "HELLO WORLD", "title": "Hello World"
}
assert apply_all([abs, str], -5) == {"abs": 5, "str": "-5"}
assert apply_all([], 1) == {}
print("✅ Q6 passed")

**Why this way**

- Functions are objects like in JS: you can store them in lists, pass them in and give them new names. `yell = shout` copies the reference and calls nothing.
- `str.upper` is the method taken from the class. `str.upper("hi")` is the same as `"hi".upper()`, which is why it works here.
- `type(type)` is `type`: classes are objects too, and their type is `type`. More on this on Day 17.

---
## Q7. Chained comparisons and `and`/`or`  ·  _Core_

**Predict first:** `and`/`or` return one of their operands, not necessarily `True`/`False`. What does each line print?

**Then write** `grade(score)`:
- raise `ValueError` if `score` is not between 0 and 100 inclusive
- `"A"` for 90+, `"B"` for 75–89, `"C"` for 50–74, `"F"` below 50

Use a chained comparison (`a <= x <= b`) for the range check.

In [ ]:
print(0 or "default")
print("a" and "b")
print("" and "b")
print(None or 0 or "")
print([] and 1 / 0)     # does this crash?
print(not [])
print(1 < 2 < 3, 3 > 2 > 1, 1 < 3 < 2)

In [ ]:
def grade(score):
    if not 0 <= score <= 100:
        raise ValueError(f"score must be 0-100, got {score}")
    if score >= 90:
        return "A"
    if score >= 75:
        return "B"
    if score >= 50:
        return "C"
    return "F"

In [ ]:
assert grade(100) == "A" and grade(90) == "A"
assert grade(89) == "B" and grade(75) == "B"
assert grade(74) == "C" and grade(50) == "C"
assert grade(49) == "F" and grade(0) == "F"
for bad in (-1, 101):
    try:
        grade(bad)
        assert False, f"grade({bad}) should raise"
    except ValueError:
        pass
print("✅ Q7 passed")

**Why this way**

- `0 <= score <= 100` means `0 <= score and score <= 100`. In JS, `0 <= x <= 100` would compare `true <= 100`.
- `[] and 1 / 0` returns `[]` without evaluating `1 / 0`. Short-circuiting works the same as in JS.
- `None or 0 or ""` returns the last operand, `""`, when everything is falsy.
- Since every branch `return`s, plain `if`s work and `elif`/`else` aren't needed. Both styles are fine.
- Put the bad value in the error message (`got {score}`). It makes debugging easier.

---
## Q8. Tuple assignment: Fibonacci  ·  _Core_

Python can assign several names at once: `a, b = b, a` swaps two values without a temp variable.

Write `fib(n)` returning the n-th Fibonacci number (`fib(0) = 0`, `fib(1) = 1`) **iteratively**, using tuple assignment. Aim for 4 lines in the body.

Then look at `fib(100)`. In JS you'd need `BigInt` for it.

In [ ]:
def fib(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

In [ ]:
assert [fib(i) for i in range(10)] == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
assert fib(100) == 354224848179261915075
print(fib(100))
print("✅ Q8 passed")

**Why this way**

- The right-hand side `b, a + b` is evaluated completely **before** assigning, so no temp variable is needed.
- `_` is the convention for "I don't use this loop variable".
- `range(n)` gives 0..n-1 lazily, like `for (let i = 0; i < n; i++)`.
- `fib(100)` is `354224848179261915075`. Python `int`s grow as needed with no overflow.

---
## Q9. Scope gotchas  ·  _Challenge_

No code to write: **predict the output** of each snippet, then run it. In the markdown cell below, write one sentence per snippet explaining *why*. Then fix snippet C so it prints `10` and then `20`.

In [ ]:
# Snippet A
for i in range(3):
    pass
print("A:", i)

# Snippet B
if True:
    secret = "visible"
print("B:", secret)

# Snippet C
x = 10
def show():
    try:
        print("C:", x)
    except UnboundLocalError as e:
        print("C: UnboundLocalError ->", e)
    x = 20
show()

In [ ]:
x = 10
def show():
    print(x)        # reads the global x
    local_x = 20    # a new local name, so x is never local
    print(local_x)

show()

**Why this way**

- **A:** Python has no block scope. The loop variable stays alive after the loop with its last value, `2`. (JS `let i` would be gone.)
- **B:** `if` doesn't create a scope either. Only functions, classes and modules do. (Comprehensions also get their own scope, Day 6.)
- **C:** Python decides at *compile time* that `x` is local to `show`, because it is assigned somewhere in the function. So `print(x)` reads a local that hasn't been set yet. JS would give a similar error with `let` (the "temporal dead zone").
- To actually change the global, you'd write `global x` at the top of the function. Usually that's a sign the function should `return` a value instead (Day 8).

---
## Q10. `safe_get`: optional chaining in Python  ·  _Challenge_

Python has no `?.` or `??`. Write `safe_get(data, path, default=None)` that does what this JS does:

```js
data?.user?.tags?.[1] ?? defaultValue
```

- `path` is a dot-separated string, e.g. `"user.tags.1"`
- dict segments are keys; for lists, a segment of digits is an index
- if any step is missing, out of range, or not traversable (e.g. indexing into a string or `None`), return `default`
- like `??`, **only `None`** falls back to `default`. Falsy values like `0`, `False` and `""` are returned as they are

Hints: `isinstance(x, dict)`, `isinstance(x, list)`, `"12".isdigit()`, `key in some_dict`.

In [ ]:
def safe_get(data, path, default=None):
    current = data
    for key in path.split("."):
        if isinstance(current, dict) and key in current:
            current = current[key]
        elif isinstance(current, list) and key.isdigit() and int(key) < len(current):
            current = current[int(key)]
        else:
            return default
    return default if current is None else current


# EAFP version ("easier to ask forgiveness than permission"), a common Python style (Day 12):
def safe_get_eafp(data, path, default=None):
    current = data
    try:
        for key in path.split("."):
            current = current[int(key)] if isinstance(current, list) else current[key]
    except (KeyError, IndexError, TypeError, ValueError):
        return default
    return default if current is None else current

In [ ]:
data = {"user": {"name": "Asha", "age": 0, "active": False, "tags": ["admin", "beta"], "address": None}}
assert safe_get(data, "user.name") == "Asha"
assert safe_get(data, "user.age", 99) == 0              # falsy, but not None
assert safe_get(data, "user.active", True) is False
assert safe_get(data, "user.tags.1") == "beta"
assert safe_get(data, "user.tags.5", "none") == "none"   # out of range
assert safe_get(data, "user.tags.x", "none") == "none"   # not an index
assert safe_get(data, "user.address", "n/a") == "n/a"    # None -> default
assert safe_get(data, "user.address.city", "?") == "?"   # can't go into None
assert safe_get(data, "user.name.first", "?") == "?"     # can't go into a str
assert safe_get(data, "nope") is None
print("✅ Q10 passed")

**Why this way**

- `isinstance` checks the type before indexing. That's "look before you leap" (LBYL).
- The EAFP version just tries it and catches the errors. Python code often prefers this, and it handles odd cases without extra `if`s. One catch: on a list, `int("-1")` succeeds and returns the *last* item, so the LBYL version is stricter here.
- The last line is the `??` part: `default if current is None else current`. Writing `current or default` would wrongly replace `0`, `False` and `""`. That's the `||` vs `??` difference.
- `str.split(".")` returns a list of segments, like JS `split`.